# Visualización de datos (Matplotlib, Seaborn, Plotly y Folium)

Este notebook reproduce las visualizaciones del reporte en PDF.

**Datasets (datos abiertos):**
- *BD_Completa.csv* (Gobierno de Puebla): indicadores con columnas por año (2010–2020).
- *estaciones_climatologicas_sih2.csv* (datos.gob.mx): estaciones climatológicas con latitud, longitud y altitud.

> Nota: el notebook descarga los CSV desde sus URLs. Si no tienes internet, puedes colocar los archivos en la carpeta `data/`.


In [ ]:
# --- Imports ---
import os
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import folium
from IPython.display import IFrame, display

sns.set_theme(style="whitegrid")


## 1) Descargar / cargar datasets

In [ ]:
# URLs de datos abiertos
URL_BD_COMPLETA = "https://sped.puebla.gob.mx/docs/datos-abiertos/2019-2024/ped/datos-generales/BD_Completa.csv"
URL_ESTACIONES = "https://www.datos.gob.mx/dataset/ea2fb2bc-3974-4025-b814-a8f3e4443b67/resource/2c135da3-118e-43f3-be63-39c5c3928b61/download/estaciones_climatologicas_sih2.csv"

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

bd_path = DATA_DIR / "BD_Completa.csv"
est_path = DATA_DIR / "estaciones_climatologicas_sih2.csv"

# Descargar solo si no existe
if not bd_path.exists():
    print("Descargando BD_Completa.csv ...")
    bd = pd.read_csv(URL_BD_COMPLETA)
    bd.to_csv(bd_path, index=False)
else:
    bd = pd.read_csv(bd_path)

if not est_path.exists():
    print("Descargando estaciones_climatologicas_sih2.csv ...")
    est = pd.read_csv(URL_ESTACIONES)
    est.to_csv(est_path, index=False)
else:
    est = pd.read_csv(est_path)

print("BD_Completa:", bd.shape)
print("Estaciones:", est.shape)
bd.head()


## 2) Preparación de datos para visualizaciones económicas (Puebla)

Tomaremos tres indicadores:
- **Tasa de desempleo** (No. Indicador = 30)
- **Valor del PIB real** (No. Indicador = 687)
- **% de personas en situación de pobreza** (No. Indicador = 85)

Y, para pobreza regional, usaremos las filas con `Cobertura = Regional` y el indicador de pobreza.


In [ ]:
years = [str(y) for y in range(2010, 2021)]

def series_from_row(row, years):
    # Convierte las columnas por año a una serie numérica
    s = row[years].replace({',': ''}, regex=True)
    s = pd.to_numeric(s, errors='coerce')
    return pd.Series(s.values, index=pd.Index([int(y) for y in years], name='Year'), name=str(row['Indicador']))

unemp_row = bd[bd["No. Indicador"] == 30].iloc[0]
gdp_row   = bd[bd["No. Indicador"] == 687].iloc[0]
pov_row   = bd[bd["No. Indicador"] == 85].iloc[0]

unemp_ts = series_from_row(unemp_row, years)
gdp_ts   = series_from_row(gdp_row, years)
pov_ts   = series_from_row(pov_row, years)

unemp_ts, gdp_ts.head(), pov_ts.head()


## 3) Visualización estática (Matplotlib + Seaborn)

Se generan 4 gráficos:
1. Serie temporal (línea).
2. Dispersión (scatter).
3. Boxplot (caja).
4. Barras.


In [ ]:
# Carpeta de salida
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)


### 3.1 Serie temporal (línea) - Tasa de desempleo

In [ ]:
plt.figure(figsize=(8,4.5))
plt.plot(unemp_ts.index, unemp_ts.values, marker='o', linewidth=2)
plt.title("Puebla - Tasa de desempleo (2010-2020)")
plt.xlabel("Año")
plt.ylabel("Tasa de desempleo (%)")
plt.tight_layout()
plt.show()

# exportar
line_path = OUT_DIR / "01_line_unemployment.png"
plt.figure(figsize=(8,4.5))
plt.plot(unemp_ts.index, unemp_ts.values, marker='o', linewidth=2)
plt.title("Puebla - Tasa de desempleo (2010-2020)")
plt.xlabel("Año")
plt.ylabel("Tasa de desempleo (%)")
plt.tight_layout()
plt.savefig(line_path, dpi=150)
plt.close()
print("Guardado:", line_path)


### 3.2 Dispersión (scatter) - PIB real vs desempleo

In [ ]:
scatter_df = pd.DataFrame({
    "Año": unemp_ts.index,
    "Tasa de desempleo (%)": unemp_ts.values,
    "PIB real (millones de pesos)": gdp_ts.values
}).dropna()

plt.figure(figsize=(8,4.8))
ax = sns.scatterplot(data=scatter_df, x="PIB real (millones de pesos)", y="Tasa de desempleo (%)", s=70)
for _, r in scatter_df.iterrows():
    ax.text(r["PIB real (millones de pesos)"], r["Tasa de desempleo (%)"], str(int(r["Año"])), fontsize=8)
plt.title("Relación entre PIB real y tasa de desempleo (Puebla, 2010-2020)")
plt.tight_layout()
plt.show()

scatter_path = OUT_DIR / "02_scatter_unemp_vs_gdp.png"
plt.figure(figsize=(8,4.8))
ax = sns.scatterplot(data=scatter_df, x="PIB real (millones de pesos)", y="Tasa de desempleo (%)", s=70)
for _, r in scatter_df.iterrows():
    ax.text(r["PIB real (millones de pesos)"], r["Tasa de desempleo (%)"], str(int(r["Año"])), fontsize=8)
plt.title("Relación entre PIB real y tasa de desempleo (Puebla, 2010-2020)")
plt.tight_layout()
plt.savefig(scatter_path, dpi=150)
plt.close()
print("Guardado:", scatter_path)


### 3.3 Boxplot (caja) - Distribución regional de pobreza

In [ ]:
# Filtrar pobreza regional (Puebla)
pov_reg = bd[
    (bd["Indicador"].astype(str).str.strip().str.lower() == "porcentaje de personas en situación de pobreza".lower())
    & (bd["Cobertura"] == "Regional")
].copy()

# En este dataset regional, típicamente hay valores en 2010, 2015 y 2020
pov_reg_long = pov_reg.melt(id_vars=["Temática"], value_vars=["2010","2015","2020"],
                            var_name="Año", value_name="Pobreza (%)")
pov_reg_long["Pobreza (%)"] = pd.to_numeric(pov_reg_long["Pobreza (%)"], errors="coerce")

plt.figure(figsize=(8,4.8))
sns.boxplot(data=pov_reg_long, x="Año", y="Pobreza (%)")
plt.title("Distribución regional del porcentaje de pobreza (Puebla)")
plt.tight_layout()
plt.show()

box_path = OUT_DIR / "03_boxplot_regional_poverty.png"
plt.figure(figsize=(8,4.8))
sns.boxplot(data=pov_reg_long, x="Año", y="Pobreza (%)")
plt.title("Distribución regional del porcentaje de pobreza (Puebla)")
plt.tight_layout()
plt.savefig(box_path, dpi=150)
plt.close()
print("Guardado:", box_path)


### 3.4 Barras - Top 10 regiones con mayor pobreza (2020)

In [ ]:
pov_2020 = pov_reg[["Temática","2020"]].copy()
pov_2020["Pobreza 2020 (%)"] = pd.to_numeric(pov_2020["2020"], errors="coerce")
pov_2020 = pov_2020.sort_values("Pobreza 2020 (%)", ascending=False).head(10)

plt.figure(figsize=(8,5.2))
sns.barplot(data=pov_2020, y="Temática", x="Pobreza 2020 (%)")
plt.title("Top 10 regiones con mayor pobreza (Puebla, 2020)")
plt.tight_layout()
plt.show()

bar_path = OUT_DIR / "04_bar_top_regions_poverty_2020.png"
plt.figure(figsize=(8,5.2))
sns.barplot(data=pov_2020, y="Temática", x="Pobreza 2020 (%)")
plt.title("Top 10 regiones con mayor pobreza (Puebla, 2020)")
plt.tight_layout()
plt.savefig(bar_path, dpi=150)
plt.close()
print("Guardado:", bar_path)


## 4) Visualización interactiva con Plotly

Se generan:
- Serie temporal interactiva (hover y zoom)
- Diagrama de burbujas (X=PIB, Y=desempleo, tamaño=pobreza %)

Se guardan como HTML.


In [ ]:
plotly_line = px.line(
    x=unemp_ts.index, y=unemp_ts.values,
    labels={"x":"Año", "y":"Tasa de desempleo (%)"},
    title="Puebla - Tasa de desempleo (2010-2020)"
)
plotly_line.update_traces(mode="lines+markers")
plotly_line.show()

plotly_line_path = OUT_DIR / "plotly_timeseries_unemployment.html"
plotly_line.write_html(plotly_line_path, include_plotlyjs="cdn")
print("Guardado:", plotly_line_path)


In [ ]:
# Pobreza estatal tiene años sin medición; se interpola para usarla como tamaño de burbuja
pov_filled = pov_ts.interpolate().ffill().bfill()

bubble_df = pd.DataFrame({
    "Año": unemp_ts.index.astype(int),
    "PIB real (millones de pesos)": gdp_ts.values,
    "Tasa de desempleo (%)": unemp_ts.values,
    "Pobreza (%) (interp.)": pov_filled.values
}).dropna()

plotly_bubble = px.scatter(
    bubble_df,
    x="PIB real (millones de pesos)",
    y="Tasa de desempleo (%)",
    size="Pobreza (%) (interp.)",
    color="Año",
    hover_data=["Año","Pobreza (%) (interp.)"],
    title="PIB vs Desempleo (tamaño = pobreza %, interpolada)"
)
plotly_bubble.show()

plotly_bubble_path = OUT_DIR / "plotly_bubble_gdp_unemployment_poverty.html"
plotly_bubble.write_html(plotly_bubble_path, include_plotlyjs="cdn")
print("Guardado:", plotly_bubble_path)


## 5) Visualización geoespacial con Folium

Usamos el dataset de estaciones climatológicas y filtramos a **Nuevo León**.
- Círculos proporcionales a **altitud**
- Popups con información


In [ ]:
nl = est[est["estado"].astype(str).str.strip().str.lower() == "nuevo león"].copy()
nl["altitud"] = pd.to_numeric(nl["altitud"], errors="coerce")
nl["latitud"] = pd.to_numeric(nl["latitud"], errors="coerce")
nl["longitud"] = pd.to_numeric(nl["longitud"], errors="coerce")
nl = nl.dropna(subset=["latitud","longitud"])

center = [nl["latitud"].mean(), nl["longitud"].mean()]
m = folium.Map(location=center, zoom_start=7, tiles="OpenStreetMap")

alt = nl["altitud"].fillna(0).clip(upper=2500)
radius = 3 + (alt / 2500) * 10  # 3..13

for (idx, row), r in zip(nl.iterrows(), radius):
    popup_html = f"""<b>{row['nombre']}</b><br>
Municipio: {row['municipio']}<br>
Altitud (m): {row['altitud']}<br>
Clave: {row['clave']}"""
    folium.CircleMarker(
        location=[row["latitud"], row["longitud"]],
        radius=float(r),
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.6,
        popup=folium.Popup(popup_html, max_width=300),
    ).add_to(m)

m


In [ ]:
folium_path = OUT_DIR / "folium_estaciones_nuevo_leon.html"
m.save(folium_path)
print("Guardado:", folium_path)

# (Opcional) mostrar el HTML embebido
display(IFrame(str(folium_path), width=900, height=550))


## 6) Cierre: ideas de decisión

- Cuando la tasa de desempleo muestre repuntes: diseñar acciones de empleo (capacitación, incentivos) y medir efecto por año.
- Cuando regiones lideren pobreza: focalizar proyectos sociales y evaluar reducción en la siguiente medición.
